<!--nav--> [🗺 Learning path](README.md) · **35/39** · ◀ [Long-Context Serving](./LongContext_KV_Compression_Serving.ipynb) · [RAG & Agent Serving Patterns](./RAG_Agent_Serving_Patterns.ipynb) ▶

# Serving Mixture-of-Experts: Expert Parallelism & the Straggler Problem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/MoE_Serving_Expert_Parallelism.ipynb)

Every frontier open model of the last two years is a **Mixture of Experts**: Mixtral, DeepSeek-V3,
Qwen3-MoE, Llama 4. MoE breaks an assumption that every notebook so far has quietly relied on —
that **the tokens in a batch all use the same weights**.

That one change rewrites the roofline, the memory plan, the parallelism strategy, and introduces a
failure mode dense models simply don't have: **routing imbalance**, where one GPU gets swamped and
everyone else waits.

| Part | What you'll learn |
|---|---|
| **1** | Total vs active parameters — the two numbers, and which one governs what |
| **2** | **The MoE roofline**: why MoE decode is *more* bandwidth-starved than dense, not less |
| **3** | Where the weights go: expert parallelism, and the all-to-all it costs |
| **4** | **Routing imbalance simulated** — the straggler that decides your step time |
| **5** | Capacity factor, token dropping, and batch size (MoE's intuition is inverted) |
| **6** | Running it: memory planning, vLLM flags, and what to monitor |

**Runs on:** any CPU — all modeling and simulation.

In [ ]:
import math, json, random, uuid, statistics
from collections import defaultdict
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · Two parameter counts

A dense model has one parameter count. An MoE has two, and confusing them is the classic error:

```
                    ┌──────────────── every token goes through ─────────────────┐
   token ──► attention ──► router ──► [expert 1] [expert 2] ... [expert 256]
                              │           ▲          ▲
                              └───────────┴──────────┘  only top-k experts run (k = 1, 2, or 8)

   TOTAL params    = everything above           → decides MEMORY (all experts must be resident)
   ACTIVE params   = attention + k experts      → decides FLOPs per token
```

- **Total parameters** set your **memory** bill. Every expert must be in VRAM (or reachable fast),
  even the ones this token doesn't use.
- **Active parameters** set your **compute** bill — and, crucially, also your **weight-read** bill
  per token during decode.

That's why MoE is described as "the quality of a big model at the cost of a small one" — true for
FLOPs, **false for memory**.

In [ ]:
MOE = {
  # name              total_B active_B experts topk  layers kv_heads head_dim  note
  "Mixtral-8x7B":   dict(total=46.7, active=12.9, experts=8,   topk=2, layers=32, kvh=8, hd=128),
  "Mixtral-8x22B":  dict(total=141,  active=39,   experts=8,   topk=2, layers=56, kvh=8, hd=128),
  "DeepSeek-V3":    dict(total=671,  active=37,   experts=256, topk=8, layers=61, kvh=None, hd=None),
  "Qwen3-235B-A22B":dict(total=235,  active=22,   experts=128, topk=8, layers=94, kvh=4, hd=128),
}
DENSE = {"Llama-3.1-70B": dict(total=70, active=70), "Llama-3.1-8B": dict(total=8, active=8)}

print(f"{'model':<18}{'total':>8}{'active':>8}{'ratio':>7}{'weights fp16':>14}{'GPUs to hold':>14}")
print("-" * 70)
for name, m in list(MOE.items()) + [(k, v) for k, v in DENSE.items()]:
    gb = m["total"] * 2
    ratio = m["total"] / m["active"]
    # smallest power-of-two GPU count of 80GB cards that holds the weights with room to serve
    n = 1
    while gb * 1.25 > 80 * n and n < 64: n *= 2
    tag = "MoE" if name in MOE else "dense"
    print(f"{name:<18}{m['total']:>7.0f}B{m['active']:>7.0f}B{ratio:>7.1f}x{gb:>12.0f}GB"
          f"{n:>10} x80GB   {tag}")

print("\nDeepSeek-V3: 671B of weights you must PAY to store, 37B you actually COMPUTE with.")
print("An 18x gap. Memory planning and FLOP planning have completely decoupled.")

## Part 2 · The MoE roofline

Notebook 31 derived, for dense decode: `AI = 2B / bytes_per_weight`, where B is the batch size —
because **one weight read serves the whole batch**.

MoE breaks that sharing. With 256 experts and top-8 routing, a batch of 32 tokens might touch
**most of the experts**, each for only a handful of tokens. The weight read per expert is amortized
over far fewer tokens than the batch size suggests:

$$\text{tokens per expert} \approx \frac{B \times k}{E}\quad\text{(if routing is uniform)}$$

$$\text{AI}_{MoE} \approx \frac{2 \times \max(1, Bk/E)}{\text{bytes}}$$

**This is the single most important fact about MoE serving:** at small batch, MoE decode is *more*
memory-bound than dense, because you read many experts' weights to produce very few tokens each.
MoE needs **large batches** to be efficient — the opposite of the intuition that "fewer active
params = faster".

In [ ]:
def moe_effective_ai(batch, topk, experts, bytes_per_weight=2.0):
    tokens_per_expert = max(1.0, batch * topk / experts)
    return 2 * tokens_per_expert / bytes_per_weight

def dense_ai(batch, bytes_per_weight=2.0):
    return 2 * batch / bytes_per_weight

H100_RIDGE = 990e12 / 3.35e12       # ~295 FLOP/byte (nb 31)

print(f"Arithmetic intensity vs batch size (fp16). H100 ridge point = {H100_RIDGE:.0f} FLOP/byte.\n")
print(f"{'batch':>7}{'dense':>9}{'Mixtral 8x7B':>15}{'DeepSeek-V3':>14}{'Qwen3-235B':>13}")
print(f"{'':>7}{'(E=1)':>9}{'(E=8,k=2)':>15}{'(E=256,k=8)':>14}{'(E=128,k=8)':>13}")
print("-" * 60)
for b in (1, 8, 32, 128, 512, 2048):
    row = [dense_ai(b),
           moe_effective_ai(b, 2, 8),
           moe_effective_ai(b, 8, 256),
           moe_effective_ai(b, 8, 128)]
    print(f"{b:>7}" + "".join(f"{v:>13.0f}  " if v < 1e4 else f"{v:>13.0f}  " for v in row))

print("\nAt batch 32, DeepSeek-V3's experts each see ~1 token: AI = 1, deep in memory-bound")
print("territory, while a dense model at the same batch sits at AI = 32.")
print("\nWhat batch does each need to reach the H100 ridge?")
for name, (k, e) in [("dense", (1, 1)), ("Mixtral 8x7B", (2, 8)),
                     ("Qwen3-235B", (8, 128)), ("DeepSeek-V3", (8, 256))]:
    # solve 2*(B*k/E)/2 = ridge  ->  B = ridge * E / k
    b_star = H100_RIDGE * e / k
    print(f"  {name:<16} batch ~{b_star:>7,.0f} to become compute-bound")
print("\nThat is why MoE serving is a LARGE-BATCH game, and why MoE at batch 1")
print("(a single interactive user) is the worst case in all of LLM serving.")

## Part 3 · Where do 671B parameters live?

You cannot fit DeepSeek-V3's weights on one GPU, so the experts get distributed. Three ways, usually
combined:

| Strategy | What's split | Traffic per layer | Notes |
|---|---|---|---|
| **TP** (tensor parallel) | every matrix, including each expert | all-reduce (nb 29) | works, but every GPU stores every expert |
| **EP** (expert parallel) | experts across GPUs — GPU *i* owns a subset | **all-to-all** (tokens travel to their expert and back) | the memory-efficient option |
| **DP + EP** | replicate attention, shard experts | all-to-all | the common production shape |

**Expert parallelism's cost is an all-to-all, twice per MoE layer**: dispatch tokens to whichever
GPU owns their chosen expert, then combine the results back. Unlike TP's all-reduce (a fixed-size
collective), the all-to-all volume depends on **where the router sent things** — which is data
dependent, and therefore variable, and therefore a scheduling hazard.

In [ ]:
def ep_traffic(batch, hidden, topk, ep_size, bytes_per=2, layers=1):
    # Each selected token-expert pair travels to its GPU and the result comes back.
    per_token_pairs = topk
    frac_remote = (ep_size - 1) / ep_size                      # fraction not already local
    bytes_out = batch * per_token_pairs * hidden * bytes_per * frac_remote
    return 2 * bytes_out * layers                              # dispatch + combine

print("All-to-all traffic per decode step, DeepSeek-V3-shaped (hidden 7168, top-8, 58 MoE layers):\n")
print(f"{'EP size':>8}{'batch 32':>14}{'batch 256':>14}{'batch 2048':>14}")
print("-" * 52)
for ep in (2, 4, 8, 16, 32):
    row = [ep_traffic(b, 7168, 8, ep, layers=58) / 1e9 for b in (32, 256, 2048)]
    print(f"{ep:>8}" + "".join(f"{v:>12.2f}GB" for v in row))

print("\nAt batch 2048 and EP=8 that is tens of GB moved PER DECODE STEP - which at 20 steps/s")
print("means hundreds of GB/s of pure coordination traffic.")
print("\nThis is why MoE at scale demands NVLink/xGMI-class fabric (nb 29/31), and why")
print("'EP across nodes over Ethernet' is not a viable serving topology.")
print("\nNote the shape of the trade: bigger EP spreads MEMORY but multiplies TRAFFIC.")

## Part 4 · Routing imbalance: the straggler that sets your step time

Here is the failure mode that has no dense equivalent. The router is learned, and real traffic is
not uniform — some experts are simply more popular. In expert parallelism, **the GPU holding a hot
expert becomes the critical path**, and every other GPU waits at the next all-to-all barrier.

Your step time isn't the *average* expert load. **It's the maximum.**

Let's simulate: route tokens with varying skew, assign experts to EP ranks, and measure both the
imbalance and its effect on step time.

In [ ]:
random.seed(3)

def simulate_routing(n_tokens=4096, experts=128, topk=8, ep_size=8, skew=0.0,
                     capacity_factor=None):
    # skew=0 -> uniform routing; higher skew -> Zipf-like popularity (a few hot experts).
    weights = [1.0 / ((i + 1) ** skew) for i in range(experts)]
    total = sum(weights)
    cum, acc = [], 0.0
    for w in weights:
        acc += w / total; cum.append(acc)

    def pick():
        r = random.random()
        lo, hi = 0, experts - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if cum[mid] < r: lo = mid + 1
            else: hi = mid
        return lo

    load = [0] * experts
    for _ in range(n_tokens):
        chosen = set()
        while len(chosen) < topk:
            chosen.add(pick())
        for e in chosen:
            load[e] += 1

    dropped = 0
    if capacity_factor is not None:
        cap = int(capacity_factor * n_tokens * topk / experts)
        for e in range(experts):
            if load[e] > cap:
                dropped += load[e] - cap
                load[e] = cap

    # experts are assigned round-robin to EP ranks
    rank_load = [0] * ep_size
    for e in range(experts):
        rank_load[e % ep_size] += load[e]

    mean_rank = sum(rank_load) / ep_size
    return {"skew": skew, "expert_load": load, "rank_load": rank_load,
            "imbalance": max(rank_load) / mean_rank if mean_rank else 1.0,
            "hottest_expert": max(load), "coldest_expert": min(load),
            "dropped": dropped, "dropped_frac": dropped / (n_tokens * topk)}

print("4096 tokens, 128 experts, top-8, EP=8. Step time is set by the SLOWEST rank.\n")
print(f"{'skew':>6}{'hottest expert':>16}{'coldest':>9}{'rank imbalance':>16}{'effective step time':>21}")
print("-" * 70)
sims = []
for skew in (0.0, 0.3, 0.6, 0.9, 1.2, 1.5):
    s = simulate_routing(skew=skew)
    sims.append(s)
    print(f"{skew:>6.1f}{s['hottest_expert']:>16}{s['coldest_expert']:>9}"
          f"{s['imbalance']:>15.2f}x{s['imbalance']:>19.2f}x baseline")

print("\nAt skew 1.5 the busiest RANK does ~1.6x the average work - so every step takes 1.6x")
print("as long, and 7 of your 8 GPUs are idle for 37% of it. You paid for 8 and got ~5.")
print("\nMitigations, in the order production actually applies them:")
print("  1. more tokens per step (batch): the law of large numbers smooths routing")
print("  2. expert placement: put known-hot experts on different ranks (needs profiling)")
print("  3. expert replication: duplicate the hottest experts across ranks")
print("  4. capacity factor + dropping: cap per-expert work, accept some token drop (Part 5)")

In [ ]:
# Does batching smooth the imbalance? (the law of large numbers, applied to routing)
print("Rank imbalance vs tokens per step, at fixed skew=1.2:\n")
print(f"{'tokens/step':>12}{'rank imbalance':>17}")
print("-" * 31)
batch_sims = []
for n in (64, 256, 1024, 4096, 16384):
    s = simulate_routing(n_tokens=n, skew=1.2)
    batch_sims.append({"n": n, "imbalance": s["imbalance"]})
    print(f"{n:>12}{s['imbalance']:>16.2f}x")
print("\nImbalance shrinks as tokens/step grows - another reason MoE wants big batches (Part 2).")
print("It never reaches 1.0 though: that residue is the router's genuine preference, not noise.")

viz = {"experts": sims[0]["expert_load"], "skewed": sims[4]["expert_load"],
       "ranks_uniform": sims[0]["rank_load"], "ranks_skewed": sims[4]["rank_load"],
       "batch": batch_sims}

JS = r'''
const M = {top: 26, right: 20, bottom: 34, left: 46};
const iw = W - M.left - M.right;
const h1 = 90, gap = 26;
const svg = root.append("svg").attr("width",W).attr("height", 2*h1 + gap + M.top + M.bottom + 40);

[["experts","uniform routing (skew 0): every expert equally loved", 0, "#42a5f5"],
 ["skewed","skewed routing (skew 1.2): a few experts dominate", h1+gap, "#ef5350"]]
 .forEach(([key, title, top, col]) => {
  const g = svg.append("g").attr("transform",`translate(${M.left},${M.top+top})`);
  const arr = data[key];
  const x = d3.scaleBand().domain(d3.range(arr.length)).range([0,iw]).padding(0.08);
  const y = d3.scaleLinear().domain([0, d3.max(data.skewed)*1.1]).range([h1,0]);
  g.append("text").attr("y",-8).style("font-size","12px").style("font-weight",600).text(title);
  g.append("g").call(d3.axisLeft(y).ticks(3)).style("font-size","9px");
  g.selectAll("b").data(arr).join("rect")
   .attr("x",(d,i)=>x(i)).attr("y",d=>y(d)).attr("width",x.bandwidth())
   .attr("height",d=>h1-y(d)).attr("fill",col)
   .append("title").text((d,i)=>`expert ${i}: ${d} tokens`);
  g.append("text").attr("x",iw).attr("y",-8).attr("text-anchor","end").style("font-size","10.5px")
   .style("fill","#546e7a").text(`max ${d3.max(arr)} · min ${d3.min(arr)} tokens`);
});
svg.append("text").attr("x",M.left).attr("y", 2*h1+gap+M.top+26).style("font-size","11.5px")
   .style("fill","#546e7a")
   .text("Each bar is one expert's token load in a single step. The tall bars set the pace for everyone.");
'''
show_d3(JS, viz, height=290)

## Part 5 · Capacity factor and token dropping

Training-era MoE fixed imbalance by **capping** each expert: a `capacity_factor` limits how many
tokens an expert may accept per step, and the overflow is **dropped** (the token skips that expert,
keeping only its residual path).

For **inference** this is a much harder trade, because a dropped token is a quality regression for a
specific user's request, not a statistical blip in training. Most serving stacks therefore prefer
"drop-free" routing with imbalance absorbed by batching and placement. But the knob exists, so
here's what it buys and costs:

In [ ]:
print("Capacity factor sweep at skew 1.2 (4096 tokens, 128 experts, top-8, EP=8):\n")
print(f"{'capacity factor':>16}{'rank imbalance':>16}{'tokens dropped':>17}")
print("-" * 50)
for cf in (1.0, 1.25, 1.5, 2.0, 4.0, None):
    s = simulate_routing(skew=1.2, capacity_factor=cf)
    label = "no cap" if cf is None else f"{cf:.2f}"
    print(f"{label:>16}{s['imbalance']:>15.2f}x{s['dropped_frac']:>16.2%}")

print("\nTighter caps buy you balance and pay for it in dropped tokens.")
print("At capacity 1.0 you get near-perfect balance and drop a meaningful share of routing")
print("decisions - in serving, that is a silent quality regression nobody will attribute")
print("to a config flag six months later. Prefer batching and placement first.")

## Part 6 · Running MoE in production

**Memory planning** — the part people get wrong first:

```
VRAM needed ≈ total_params × bytes        (ALL experts resident, not just active)
            + KV cache                     (nb 21 — MoE attention is usually GQA/MLA)
            + activation & all-to-all buffers
```

For DeepSeek-V3 in fp8: ~671 GB of weights alone → a full 8×H200 or 8×MI300X node, minimum.
This is where notebook 31's capacity argument bites hardest: **VRAM per GPU determines whether you
need one node or two**, and crossing a node boundary with all-to-all traffic is brutal.

**vLLM flags:**

```bash
vllm serve deepseek-ai/DeepSeek-V3 \
  --tensor-parallel-size 8 \
  --enable-expert-parallel \        # shard experts across ranks instead of replicating
  --max-model-len 8192 \
  --gpu-memory-utilization 0.90
```

**What to monitor** (extending notebook 26's list):

| Signal | Why it matters for MoE |
|---|---|
| step time **variance** | imbalance shows up as jitter, not as a higher average |
| tokens per step | MoE efficiency is a batch-size story (Part 2) — small batches are pathological |
| all-to-all time share | if this dominates, your EP width or fabric is wrong (Part 3) |
| per-expert load (if exposed) | the direct measure of Part 4; profile before you place experts |
| `gpu_cache_usage_perc` | unchanged in meaning — MoE doesn't change KV math |

**The MoE serving checklist:**

- [ ] Budget VRAM by **total** params, FLOPs by **active** params — never mix them up
- [ ] Verify the whole model fits **inside one NVLink/xGMI node**
- [ ] Drive **large batches**; MoE at batch 1 is the worst case in serving (Part 2)
- [ ] Prefer **drop-free** routing; treat capacity-factor dropping as a last resort
- [ ] Watch step-time **variance**, not just the mean
- [ ] Re-derive the flip point with the MoE formula before assuming any dense intuition ports

## Recap

1. **Two parameter counts.** Total sets memory; active sets FLOPs. MoE decouples them by up to ~18×.
2. **MoE decode is more memory-bound than dense at the same batch**, because each expert's weight
   read is amortized over `Bk/E` tokens instead of `B`. MoE is a large-batch technology.
3. **Expert parallelism trades memory for all-to-all traffic**, and that traffic is data-dependent.
4. **Routing imbalance sets step time by the slowest rank** — the failure mode dense models don't
   have. Batch size is the first and best mitigation.
5. **Token dropping fixes balance by sacrificing quality** — acceptable in training, rarely in
   serving.

### Further reading
- [Switch Transformer](https://arxiv.org/abs/2101.03961) (capacity factor, dropping) · [GShard](https://arxiv.org/abs/2006.16668)
- [Mixtral of Experts](https://arxiv.org/abs/2401.04088) · [DeepSeek-V3](https://arxiv.org/abs/2412.19437) (auxiliary-loss-free load balancing)
- [MegaBlocks](https://arxiv.org/abs/2211.15841) — dropless MoE via block-sparse kernels
- [vLLM expert parallelism docs](https://docs.vllm.ai/en/latest/serving/distributed_serving.html)
- Prerequisites here: nb [31](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) (the roofline this extends) and [29](./Distributed_MultiReplica_Serving.ipynb) (the parallelism it builds on)